# Lesson 6: K-Means Clustering

Discovering groups in unlabeled data.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import make_blobs, load_iris

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

## 1. K-Means on Synthetic Data

In [ ]:
X, y_true = make_blobs(n_samples=300, centers=4, cluster_std=0.8, random_state=42)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for i, K in enumerate([2, 3, 4, 5, 6, 8]):
    kmeans = KMeans(n_clusters=K, random_state=42, n_init=10)
    y_pred = kmeans.fit_predict(X)
    ax = axes[i // 3, i % 3]
    ax.scatter(X[:, 0], X[:, 1], c=y_pred, cmap='viridis', alpha=0.6)
    ax.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1],
               c='red', marker='x', s=200, linewidths=3)
    ax.set_title(f'K = {K}')
plt.tight_layout()
plt.show()

## 2. Elbow Method and Silhouette Score

In [ ]:
inertias, sil_scores = [], []
K_range = range(2, 11)

for K in K_range:
    kmeans = KMeans(n_clusters=K, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X)
    inertias.append(kmeans.inertia_)
    sil_scores.append(silhouette_score(X, labels))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(K_range, inertias, 'o-'); axes[0].set_xlabel('K'); axes[0].set_ylabel('Inertia')
axes[0].set_title('Elbow Method'); axes[0].grid(True)
axes[1].plot(K_range, sil_scores, 'o-'); axes[1].set_xlabel('K'); axes[1].set_ylabel('Silhouette')
axes[1].set_title('Silhouette Score'); axes[1].grid(True)
plt.tight_layout()
plt.show()

best_K = K_range[np.argmax(sil_scores)]
print(f"Optimal K by silhouette: {best_K}")

## 3. Iris Clustering (Unsupervised)

In [ ]:
iris = load_iris()
X_iris = StandardScaler().fit_transform(iris.data)

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
iris_labels = kmeans.fit_predict(X_iris)

print(pd.crosstab(iris.target, iris_labels, rownames=['True'], colnames=['Cluster']))
print(f"Silhouette: {silhouette_score(X_iris, iris_labels):.3f}")

## 4. Importance of Feature Scaling

In [ ]:
X_unscaled = np.column_stack([X[:, 0] * 100, X[:, 1]])  # First feature has larger scale

kmeans_bad = KMeans(n_clusters=4, random_state=42, n_init=10)
labels_bad = kmeans_bad.fit_predict(X_unscaled)

kmeans_good = KMeans(n_clusters=4, random_state=42, n_init=10)
labels_good = kmeans_good.fit_predict(StandardScaler().fit_transform(X_unscaled))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(X_unscaled[:, 0], X_unscaled[:, 1], c=labels_bad, cmap='viridis')
axes[0].set_title('Without Scaling')
axes[1].scatter(X_unscaled[:, 0], X_unscaled[:, 1], c=labels_good, cmap='viridis')
axes[1].set_title('With Scaling')
plt.tight_layout()
plt.show()

## 5. Biotechnology: Patient Subtypes

In [ ]:
n_p, n_g = 200, 500
X_sub = np.random.randn(n_p, n_g)
X_sub[:70, :50] += 0.5
X_sub[70:130, 50:100] += 0.5
X_sub[130:, 100:150] += 0.5

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
sub_clusters = kmeans.fit_predict(X_sub)

true_s = np.array([0]*70 + [1]*60 + [2]*70)
print(pd.crosstab(true_s, sub_clusters, rownames=['True'], colnames=['Cluster']))

## 6. SaaS: Customer Segmentation

In [ ]:
n_c = 500
customers = pd.DataFrame({
    'spend': np.random.exponential(1000, n_c),
    'frequency': np.random.poisson(12, n_c),
    'order_value': np.random.normal(50, 20, n_c),
    'tenure': np.random.exponential(24, n_c),
})

X_cust = StandardScaler().fit_transform(customers)
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
customers['segment'] = kmeans.fit_predict(X_cust)

segments = customers.groupby('segment').mean().round(1)
print(segments)
print(f"\nSilhouette: {silhouette_score(X_cust, customers['segment']):.3f}")

## 7. Exercises

### Level 1
What is the difference between supervised and unsupervised learning?

### Level 2
Generate `make_blobs(n_samples=500, centers=5)` and find optimal K using elbow + silhouette.

### Level 3
Your clustering finds 3 patient groups. How would you validate they are biologically real?

In [ ]:
# Level 2 code

## 8. Coding Challenge

Write `optimal_k(X, max_k=10)` returning optimal K from both methods.

In [ ]:
def optimal_k(X, max_k=10):
    inertias, sil_scores = [], []
    K_range = range(2, max_k + 1)
    for K in K_range:
        kmeans = KMeans(n_clusters=K, random_state=42, n_init=10)
        labels = kmeans.fit_predict(X)
        inertias.append(kmeans.inertia_)
        sil_scores.append(silhouette_score(X, labels))

    # Elbow: find K where improvement drops
    diffs = np.diff(inertias)
    elbow = K_range[np.argmin(diffs)]

    # Silhouette: max score
    sil_best = K_range[np.argmax(sil_scores)]

    print(f"Elbow suggests: K={elbow}")
    print(f"Silhouette suggests: K={sil_best}")

    if elbow == sil_best:
        print("Both methods agree!")
    else:
        print("Methods disagree — consider domain knowledge")

    return elbow, sil_best

optimal_k(X)

## Summary

- K-Means groups data without labels
- Minimizes inertia (within-cluster distances)
- Elbow + silhouette for choosing K
- Always scale features
- Useful for patient subtypes, customer segments
- Results depend on initialization (use n_init)